# 🍽️ Exploratory Data Analysis — Zomato Bangalore Restaurants

**Dataset:** [Zomato Bangalore Restaurants — Kaggle](https://www.kaggle.com/datasets/himanshupoddar/zomato-bangalore-restaurants)  
**Author:** Your Name  
**Date:** April 2026  

---

## 📌 Project Overview

Zomato is one of India's largest food-tech platforms, operating in hundreds of cities. This notebook performs a **deep Exploratory Data Analysis (EDA)** on restaurant data from **Bangalore**, India's tech capital and food hub.

### 🎯 Objectives
- Understand the structure and quality of the dataset
- Clean and preprocess the data for analysis
- Uncover meaningful insights about Bangalore's restaurant ecosystem
- Visualize trends using professional-grade charts

### 🔑 Key Questions We'll Answer
1. Which restaurant types and cuisines dominate Bangalore?
2. How does cost relate to ratings across different localities?
3. Which locations have the highest concentration of top-rated restaurants?
4. Does online ordering availability impact customer ratings?
5. What are the most popular cuisine combinations?

---
## 1️⃣ Import Libraries

We start by importing all necessary libraries:
- **pandas** — data manipulation and analysis
- **numpy** — numerical operations
- **matplotlib** — base plotting library
- **seaborn** — statistical data visualization built on matplotlib
- **warnings** — suppress unnecessary warnings for a clean output

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# ── Plot aesthetics ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'figure.facecolor': '#F9F5F0',
    'axes.facecolor': '#F9F5F0',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'font.family': 'DejaVu Sans',
})

PALETTE = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A', '#F4A261', '#264653']
sns.set_palette(PALETTE)

print("✅ All libraries imported successfully.")
print(f"   pandas  {pd.__version__}")
print(f"   numpy   {np.__version__}")
print(f"   seaborn {sns.__version__}")

---
## 2️⃣ Data Loading

We load the dataset using `pd.read_csv()`. The file is assumed to be downloaded from Kaggle and placed in the working directory.

> 📂 **Source:** `zomato.csv` — Zomato Bangalore Restaurants dataset

In [ ]:
df = pd.read_csv('zomato.csv')

print(f"📊 Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\n🔍 First 5 rows:")
df.head()

In [ ]:
print("📋 Column names and data types:")
df.dtypes

In [ ]:
print("❓ Missing values per column:")
df.isnull().sum().sort_values(ascending=False)

---
## 3️⃣ Data Cleaning

Raw data is rarely analysis-ready. This section addresses:
- **Duplicate rows** — removing exact copies
- **Missing values** — dropping or imputing where appropriate
- **Data type fixes** — converting strings to numeric
- **Column renaming** — for clarity and consistency
- **Noise removal** — stripping special characters from numeric-like columns

### 3.1 Rename Columns
Standardise column names to `snake_case` for easy access.

In [ ]:
df.rename(columns={
    'approx_cost(for two people)': 'cost_for_two',
    'listed_in(type)': 'listed_type',
    'listed_in(city)': 'listed_city',
}, inplace=True)

print("✅ Columns renamed.")
print(df.columns.tolist())

### 3.2 Remove Duplicates
Duplicate entries skew aggregations and inflate counts.

In [ ]:
before = len(df)
df.drop_duplicates(inplace=True)
after = len(df)
print(f"🗑️  Removed {before - after:,} duplicate rows. ({before:,} → {after:,})")

### 3.3 Clean the `rate` Column
The `rate` column contains values like `"4.1/5"` and `"NEW"`. We strip the `"/5"` suffix and cast to float.

In [ ]:
df['rate'] = df['rate'].astype(str).str.replace('/5', '', regex=False).str.strip()
df['rate'] = pd.to_numeric(df['rate'], errors='coerce')

print(f"✅ Rate column cleaned. Unique NaN count: {df['rate'].isna().sum():,}")
df['rate'].describe()

### 3.4 Clean the `cost_for_two` Column
This column has commas (e.g., `"1,200"`). We remove them and cast to numeric.

In [ ]:
df['cost_for_two'] = df['cost_for_two'].astype(str).str.replace(',', '', regex=False)
df['cost_for_two'] = pd.to_numeric(df['cost_for_two'], errors='coerce')

print(f"✅ cost_for_two cleaned. NaN count: {df['cost_for_two'].isna().sum():,}")
df['cost_for_two'].describe()

### 3.5 Handle Missing Values

**Strategy:**
- Drop rows where **`rate`** is missing — it's our primary analysis metric
- Fill missing **`cost_for_two`** with the median (robust to outliers)
- Fill missing **`cuisines`** with `'Unknown'`
- Drop rows with missing **`location`** (very few, not imputable)

In [ ]:
df.dropna(subset=['rate'], inplace=True)
df['cost_for_two'].fillna(df['cost_for_two'].median(), inplace=True)
df['cuisines'].fillna('Unknown', inplace=True)
df.dropna(subset=['location'], inplace=True)

print(f"✅ Missing values handled. Final shape: {df.shape}")
print("\n📉 Remaining nulls:")
df.isnull().sum()[df.isnull().sum() > 0]

### 3.6 Fix Data Types

In [ ]:
bool_map = {'Yes': True, 'No': False}
df['online_order'] = df['online_order'].map(bool_map)
df['book_table']   = df['book_table'].map(bool_map)

print("✅ Boolean columns converted.")
df[['online_order', 'book_table']].value_counts()

### 3.7 Summary After Cleaning

In [ ]:
print("=" * 50)
print("📦 CLEAN DATASET SUMMARY")
print("=" * 50)
print(f"  Rows        : {df.shape[0]:,}")
print(f"  Columns     : {df.shape[1]}")
print(f"  Restaurants : {df['name'].nunique():,} unique")
print(f"  Locations   : {df['location'].nunique()} unique")
print(f"  Avg Rating  : {df['rate'].mean():.2f} / 5")
print(f"  Avg Cost(2) : ₹{df['cost_for_two'].mean():.0f}")
print("=" * 50)

---
## 4️⃣ Exploratory Data Analysis (EDA)

Now that the data is clean, we explore it systematically — from summary statistics to grouped insights.

### 4.1 Summary Statistics

In [ ]:
df[['rate', 'votes', 'cost_for_two']].describe().round(2)

### 4.2 💡 Insight 1 — Online Ordering Impact on Ratings

> **Question:** Do restaurants that accept online orders tend to receive higher ratings?

This is a critical business question — if online ordering correlates with better ratings, it may reflect better customer reach and service.

In [ ]:
insight1 = df.groupby('online_order')['rate'].agg(['mean', 'median', 'count'])
insight1.index = ['No Online Order', 'Accepts Online Order']
insight1.columns = ['Mean Rating', 'Median Rating', 'Count']
insight1 = insight1.round(3)

print("📊 Insight 1: Online Order vs Rating")
print(insight1)
diff = insight1.loc['Accepts Online Order', 'Mean Rating'] - insight1.loc['No Online Order', 'Mean Rating']
print(f"\n🔍 Restaurants WITH online ordering score {diff:+.3f} points higher on average.")

### 4.3 💡 Insight 2 — Top Locations by Average Rating

> **Question:** Which Bangalore localities consistently produce the highest-rated dining experiences?

We filter to locations with at least **50 restaurants** to avoid small-sample bias.

In [ ]:
location_stats = (
    df.groupby('location')
    .agg(avg_rating=('rate', 'mean'), restaurant_count=('name', 'count'), avg_cost=('cost_for_two', 'mean'))
    .query('restaurant_count >= 50')
    .sort_values('avg_rating', ascending=False)
    .round(2)
)

print("📊 Insight 2: Top 10 Locations by Avg Rating (min 50 restaurants)")
location_stats.head(10)

### 4.4 💡 Insight 3 — Cost vs Rating Segment Analysis

> **Question:** Are expensive restaurants necessarily better rated?

We segment restaurants into budget tiers and compare average ratings across segments.

In [ ]:
bins   = [0, 300, 600, 1000, 2000, 10000]
labels = ['Budget\n(<₹300)', 'Affordable\n(₹300–600)', 'Mid-Range\n(₹600–1k)', 'Premium\n(₹1k–2k)', 'Luxury\n(₹2k+)']

df['cost_segment'] = pd.cut(df['cost_for_two'], bins=bins, labels=labels)

segment_stats = df.groupby('cost_segment', observed=False).agg(
    avg_rating=('rate', 'mean'),
    count=('name', 'count')
).round(3)

print("📊 Insight 3: Rating by Cost Segment")
segment_stats

### 4.5 💡 Insight 4 — Most Popular Cuisines

In [ ]:
cuisine_series = df['cuisines'].str.split(',').explode().str.strip()
top_cuisines   = cuisine_series.value_counts().head(15)

print("📊 Insight 4: Top 15 Cuisines by Frequency")
print(top_cuisines.to_string())

### 4.6 💡 Insight 5 — Restaurant Type Distribution & Ratings

In [ ]:
type_stats = (
    df.groupby('rest_type')
    .agg(avg_rating=('rate','mean'), count=('name','count'))
    .query('count >= 100')
    .sort_values('avg_rating', ascending=False)
    .round(2)
)

print("📊 Insight 5: Restaurant Types (min 100) by Rating")
type_stats

---
## 5️⃣ Data Visualization

The following section presents **6 professional visualizations** that tell a cohesive data story about Bangalore's restaurant scene.

### 📊 Plot 1 — Bar Plot: Top 10 Most Common Restaurant Types

In [ ]:
top_types = df['rest_type'].value_counts().head(10)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top_types.index[::-1], top_types.values[::-1],
               color=PALETTE[0], edgecolor='none', height=0.65)

# Annotate
for bar in bars:
    ax.text(bar.get_width() + 80, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():,.0f}', va='center', fontsize=10, color='#333')

ax.set_title('Top 10 Most Common Restaurant Types in Bangalore', pad=15)
ax.set_xlabel('Number of Restaurants')
ax.set_xlim(0, top_types.max() * 1.15)
ax.axvline(top_types.mean(), color='#264653', linestyle='--', linewidth=1.2, label=f'Mean = {top_types.mean():,.0f}')
ax.legend(fontsize=9)
fig.tight_layout()
plt.savefig('plot1_restaurant_types.png', dpi=150, bbox_inches='tight')
plt.show()
print("💡 Quick Delivery and Casual Dining dominate Bangalore's restaurant landscape.")

### 📊 Plot 2 — Count Plot: Online Order Availability by Restaurant Type

In [ ]:
top5_types = df['rest_type'].value_counts().head(5).index
df_top5 = df[df['rest_type'].isin(top5_types)].copy()
df_top5['Online Order'] = df_top5['online_order'].map({True: 'Yes', False: 'No'})

fig, ax = plt.subplots(figsize=(12, 6))
sns.countplot(data=df_top5, x='rest_type', hue='Online Order',
              palette=[PALETTE[0], PALETTE[1]], ax=ax, edgecolor='none',
              order=top5_types)

ax.set_title('Online Order Availability Across Top 5 Restaurant Types', pad=15)
ax.set_xlabel('Restaurant Type')
ax.set_ylabel('Count')
ax.legend(title='Online Order', title_fontsize=10)
ax.tick_params(axis='x', rotation=15)
fig.tight_layout()
plt.savefig('plot2_online_order_countplot.png', dpi=150, bbox_inches='tight')
plt.show()
print("💡 Quick Bites overwhelmingly offer online ordering, while Casual Dining is more split.")

### 📊 Plot 3 — Box Plot: Rating Distribution by Cost Segment

In [ ]:
df_seg = df.dropna(subset=['cost_segment'])

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df_seg, x='cost_segment', y='rate',
            palette=PALETTE, ax=ax, linewidth=1.2,
            order=labels, flierprops=dict(marker='o', markersize=3, alpha=0.4))

ax.set_title('Rating Distribution Across Price Segments', pad=15)
ax.set_xlabel('Price Segment (Cost for Two People)')
ax.set_ylabel('Rating (out of 5)')
ax.axhline(df['rate'].mean(), linestyle='--', color='#264653', linewidth=1,
           label=f'Overall Mean = {df["rate"].mean():.2f}')
ax.legend(fontsize=9)
fig.tight_layout()
plt.savefig('plot3_rating_by_cost.png', dpi=150, bbox_inches='tight')
plt.show()
print("💡 Higher-priced restaurants tend to have higher median ratings, but with more variance.")

### 📊 Plot 4 — Heatmap: Correlation Between Numerical Features

In [ ]:
num_df = df[['rate', 'votes', 'cost_for_two', 'online_order', 'book_table']].copy()
num_df['online_order'] = num_df['online_order'].astype(int)
num_df['book_table']   = num_df['book_table'].astype(int)
num_df.rename(columns={'online_order': 'Online Order', 'book_table': 'Book Table',
                        'rate': 'Rating', 'votes': 'Votes', 'cost_for_two': 'Cost (2)'}, inplace=True)

corr = num_df.corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, linecolor='white', ax=ax,
            annot_kws={'size': 11}, vmin=-1, vmax=1,
            cbar_kws={'shrink': 0.8})

ax.set_title('Correlation Heatmap — Key Features', pad=15)
fig.tight_layout()
plt.savefig('plot4_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("💡 Cost & Rating show moderate positive correlation. Votes correlate with online ordering presence.")

### 📊 Plot 5 — Horizontal Bar: Top 10 Cuisines by Count

In [ ]:
top10_c = cuisine_series.value_counts().head(10)

fig, ax = plt.subplots(figsize=(11, 6))
colors = [PALETTE[i % len(PALETTE)] for i in range(len(top10_c))]
bars = ax.barh(top10_c.index[::-1], top10_c.values[::-1],
               color=colors[::-1], edgecolor='none', height=0.65)

for bar in bars:
    ax.text(bar.get_width() + 40, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():,}', va='center', fontsize=10)

ax.set_title('Top 10 Most Offered Cuisines in Bangalore', pad=15)
ax.set_xlabel('Number of Restaurants Offering')
ax.set_xlim(0, top10_c.max() * 1.15)
fig.tight_layout()
plt.savefig('plot5_top_cuisines.png', dpi=150, bbox_inches='tight')
plt.show()
print("💡 North Indian and Chinese cuisines are the most widely offered — reflecting broad Indian appetite.")

### 📊 Plot 6 — Line Plot: Average Rating Trend Across Location Rank

In [ ]:
# Top 20 locations by restaurant count, ordered by avg rating
top20_locs = (
    df.groupby('location')
    .agg(avg_rating=('rate', 'mean'), count=('name', 'count'))
    .query('count >= 30')
    .sort_values('avg_rating', ascending=False)
    .head(20)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(top20_locs.index, top20_locs['avg_rating'],
        color=PALETTE[2], linewidth=2.5, marker='o', markersize=7,
        markerfacecolor='white', markeredgewidth=2, markeredgecolor=PALETTE[2])

ax.fill_between(top20_locs.index, top20_locs['avg_rating'],
                top20_locs['avg_rating'].min() - 0.05,
                alpha=0.12, color=PALETTE[2])

# Annotate top 3
for i in range(3):
    ax.annotate(top20_locs.loc[i, 'location'],
                xy=(i, top20_locs.loc[i, 'avg_rating']),
                xytext=(i + 0.3, top20_locs.loc[i, 'avg_rating'] + 0.02),
                fontsize=8, color='#264653',
                arrowprops=dict(arrowstyle='->', color='#aaa', lw=1))

ax.set_xticks(top20_locs.index)
ax.set_xticklabels(top20_locs['location'], rotation=45, ha='right', fontsize=9)
ax.set_title('Top 20 Locations — Average Rating Trend (min 30 restaurants)', pad=15)
ax.set_ylabel('Average Rating')
ax.set_xlabel('Location (ranked by avg rating)')
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f'))

fig.tight_layout()
plt.savefig('plot6_location_rating_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print("💡 Rating drops gradually across locations — even lower-ranked areas maintain respectable scores.")

---
## 6️⃣ Key Takeaways & Conclusions

| # | Insight | Finding |
|---|---------|----------|
| 1 | **Online Ordering & Ratings** | Restaurants that accept online orders score higher on average — likely reflecting a more customer-centric approach |
| 2 | **Location Matters** | Certain localities consistently outperform others — a signal for restaurant entrepreneurs |
| 3 | **Price ≠ Quality (always)** | Premium restaurants tend to score higher, but mid-range outlets show the best value-to-rating ratio |
| 4 | **Cuisine Preferences** | North Indian & Chinese cuisines dominate — fast, affordable, and familiar |
| 5 | **Type Drives Strategy** | Quick Bites lead in count; Fine Dining leads in rating — different business models, different metrics |

---

### 🚀 Next Steps
- **Sentiment Analysis** on customer reviews using NLP
- **Geospatial mapping** of restaurant density across Bangalore
- **Predictive modelling** — can we predict a restaurant's rating from its features?
- **Time-series analysis** if order history data were available

---
*Notebook completed. All plots saved as `.png` files in the working directory.*